# Decision Layer

Converts the calibrated 24-month PD into the artefacts a credit decision is actually made
from: an approval-rate versus bad-rate curve, an LGD estimated from history, a lifetime PD,
and a cutoff table carrying expected loss.

Champion: GBM application-only, Platt-calibrated on validation, anchored to the long-run rate.
Continues from `evaluation_phase5.ipynb`; the setup below repeats its first three cells.


In [ ]:
from pathlib import Path

import numpy as np
import polars as pl
import yaml

from credit_risk.data.ingestion import load_raw_accepted_loans
from credit_risk.data.target import build_target
from credit_risk.evaluation.business import (
    approval_curve,
    cutoff_table,
    empirical_lgd,
    horizon_coverage,
    scheduled_gross_yield,
    to_lifetime_pd,
)
from credit_risk.evaluation.calibration import Calibrator, central_tendency_shift, pd_to_score
from credit_risk.features.build_dataset import application_features, assemble_feature_matrix
from credit_risk.features.cleaning import clean_features
from credit_risk.models.gbm import predict_gbm, train_gbm

pl.Config.set_tbl_rows(30)
CONFIG_PATH = Path("../configs/base.yaml")
DATA_PATH = Path("../data/raw/accepted_2007_to_2018Q4.csv")

raw = load_raw_accepted_loans(DATA_PATH)
labeled = build_target(raw, CONFIG_PATH)
# term_months is derived by clean_features, not by build_target, so `labeled` does not
# have it. `cleaned` keeps every vintage (assemble_feature_matrix drops pre-2013),
# which section 3 needs: the coverage denominator must count ALL of a vintage's
# defaults, not just those inside a split.
cleaned = clean_features(labeled)
final = assemble_feature_matrix(labeled, CONFIG_PATH)
splits = {n: final.filter(pl.col("split") == n) for n in ("train", "validation", "oot_test")}


In [ ]:
features = application_features(final)
params = yaml.safe_load(Path("../configs/gbm_best_params_application.yaml").read_text())
model, features = train_gbm(splits["train"], splits["validation"], params=params, features=features)
pred = {n: predict_gbm(model, features, df) for n, df in splits.items()}
y = {n: df["default_flag"].to_numpy() for n, df in splits.items()}

calibrator = Calibrator("platt").fit(y["validation"], pred["validation"])
pd_oot = calibrator.transform(pred["oot_test"])
long_run = float(np.mean([y[n].mean() for n in splits]))
logits = np.log(pd_oot / (1 - pd_oot)) + central_tendency_shift(pd_oot, long_run)
pd_oot = 1 / (1 + np.exp(-logits))
scores = pd_to_score(pd_oot)
print("mean PD", round(float(pd_oot.mean()), 4), " score", round(float(scores.min())), "-", round(float(scores.max())))


## 1. Approval rate versus bad rate

The central trade-off. `bad_rate` is measured on the APPROVED book at each cutoff, not on
everyone - the two diverge fast and only the first is what the portfolio will run at.
`bad_rate_declined` is what is being turned away, which is the number that defends a cutoff
against the argument that it is too tight.


In [ ]:
oot = splits["oot_test"]
ead = oot["loan_amnt"].to_numpy().astype(float)

curve = approval_curve(scores, y["oot_test"], exposure=ead, n_points=20)
print(curve.to_pandas().to_string(index=False))
curve.write_csv("../docs/approval_curve.csv")


## 2. LGD from history

Assuming an LGD is the easiest place to make an expected-loss number meaningless. This reads
it off charged-off loans: the share of still-outstanding principal never recovered.

Uses post-origination columns to estimate a portfolio parameter from history - legitimate
here, still banned as model input.


In [ ]:
lgd_stats = empirical_lgd(cleaned)
print(lgd_stats)
LGD = lgd_stats["exposure_weighted_lgd"]
print("\nusing exposure-weighted LGD:", round(LGD, 4))


## 3. From 24-month PD to lifetime PD

The model predicts default within 24 months. Expected loss over a loan's life needs the
lifetime figure, so measure - on the fully matured 2013 vintage - what share of each term's
eventual defaults actually lands inside the window, then divide by it.

The two terms will differ, and that difference is the whole point: a single blended factor
understates 60-month exposure.


In [ ]:
# Denominator: every default the 2013 vintage ever had, per term. 2013 is fully
# matured - a 2013-12 loan on a 60-month term finishes in 2018-12, the data cutoff.
matured = cleaned.filter(
    pl.col("issue_d").cast(pl.Utf8).str.strptime(pl.Date, "%b-%Y", strict=False).dt.year() == 2013
)
eventual = (
    matured.filter(pl.col("loan_status").is_in(["Charged Off", "Default"]))
    .group_by("term_months").agg(pl.len().alias("n_default_eventual"))
)
coverage = (
    horizon_coverage(cleaned, horizon=24, vintages=[2013])
    .join(eventual, on="term_months")
    .with_columns((pl.col("n_default_in_horizon") / pl.col("n_default_eventual")).alias("coverage"))
)
print(coverage)


In [ ]:
coverage_map = dict(zip(coverage["term_months"], coverage["coverage"], strict=True))
coverage_vector = oot["term_months"].replace_strict(coverage_map, default=1.0).to_numpy().astype(float)

pd_lifetime = to_lifetime_pd(pd_oot, coverage_vector)
for term in (36, 60):
    mask = oot["term_months"].to_numpy() == term
    print(f"term={term}  coverage={coverage_map[term]:.3f}  "
          f"PD24={pd_oot[mask].mean():.4f} -> PD_life={pd_lifetime[mask].mean():.4f}")


## 4. Cutoff table with expected loss

`expected_loss_rate` is the model's forecast loss per unit of approved exposure;
`realised_loss_rate` uses the observed 24-month outcome instead. They will not match - the
first is lifetime, the second is 24-month - and the ratio between them is a useful check that
the horizon correction in section 3 is the right order of magnitude.


In [ ]:
# Revenue side from the contractual terms, so a break-even cutoff is computable.
gross_yield = scheduled_gross_yield(oot)
table = cutoff_table(
    scores, y["oot_test"], pd_lifetime, ead, lgd=LGD, n_points=20, gross_yield=gross_yield,
)
print(table.select(
    "cutoff", "approval_rate", "bad_rate", "expected_loss_rate", "gross_yield_rate",
    "net_margin_rate",
).with_columns(
    (pl.col("net_margin_total") / 1e6).round(1).alias("net_margin_$M"),
    pl.col("marginal_margin_rate").round(4),
) if False else table.select(
    "approval_rate", "bad_rate", "expected_loss_rate", "gross_yield_rate",
    "net_margin_rate",
    (pl.col("net_margin_total") / 1e6).round(1).alias("net_margin_musd"),
    "marginal_margin_rate",
).to_pandas().to_string(index=False))
table.write_csv("../docs/cutoff_table.csv")


## 5. Picking a cutoff

There is no optimum without a policy target, and stating the target is the decision. Three
common framings, all readable off the table above:

- **Risk appetite:** the loosest cutoff whose approved bad rate stays under a ceiling.
- **Volume floor:** the tightest cutoff still approving a required share of applicants.
- **Break-even:** the cutoff where expected loss rate meets the portfolio's net margin.

The first is filled in below as an example. Replace the ceiling with the real appetite -
the number is an input, not something the data can supply.


In [ ]:
# Framing A - risk appetite: loosest cutoff whose approved bad rate stays under a ceiling.
BAD_RATE_CEILING = 0.06
eligible = table.filter(pl.col("bad_rate") <= BAD_RATE_CEILING).sort("approval_rate", descending=True)
if eligible.height:
    r = eligible.row(0, named=True)
    print(f"risk appetite   -> approves {r['approval_rate']:.0%}, bad {r['bad_rate']:.2%}, "
          f"margin {r['net_margin_rate']:.2%}, total {r['net_margin_total']/1e6:.0f}M")

# Framing B - marginal economics: last cutoff whose ADDED tranche still earns money.
# This is the profit-maximising answer; note it is far looser than framing C.
viable = table.filter(pl.col("marginal_margin_rate") > 0).sort("approval_rate", descending=True)
if viable.height:
    r = viable.row(0, named=True)
    print(f"marginal zero   -> approves {r['approval_rate']:.0%}, bad {r['bad_rate']:.2%}, "
          f"margin {r['net_margin_rate']:.2%}, total {r['net_margin_total']/1e6:.0f}M")

# Framing C - maximise margin PER UNIT of exposure. Shown to be contrasted, not followed:
# it optimises efficiency while ignoring volume, so it always lands too tight.
r = table.sort("net_margin_rate", descending=True).row(0, named=True)
print(f"max margin rate -> approves {r['approval_rate']:.0%}, bad {r['bad_rate']:.2%}, "
      f"margin {r['net_margin_rate']:.2%}, total {r['net_margin_total']/1e6:.0f}M")
